In [1]:
import pandas as pd
import numpy as np
import scipy.io as sio
import torch
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from data_classes.decomposition import Extract_Features
import torch.utils


In [2]:
phase1_data = sio.loadmat('../data/mine_impact_data_2019.mat')
samples  = pd.DataFrame(phase1_data["x"].T)
labels  = pd.DataFrame(phase1_data["y"].T, columns=["y"])

df = pd.concat([samples, labels], axis=1, join="inner")

df = df.dropna()

In [3]:

shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_X = shuffled_df.iloc[:, :-1]
df_Y = shuffled_df.iloc[:, -1]

data = Extract_Features(
    df_X=df_X,
    df_Y=df_Y,
    feature = "autoencoder_cnn",
    weight_decay = 0,
    n_epochs = 10,
    batch_size = 30,
    lr =  1e-2
)

Epoch [1/10], Reconstruction Loss: 0.000213
Epoch [2/10], Reconstruction Loss: 0.000060
Epoch [3/10], Reconstruction Loss: 0.000060
Epoch [4/10], Reconstruction Loss: 0.000060
Epoch [5/10], Reconstruction Loss: 0.000060
Epoch [6/10], Reconstruction Loss: 0.000060
Epoch [7/10], Reconstruction Loss: 0.000060
Epoch [8/10], Reconstruction Loss: 0.000060
Epoch [9/10], Reconstruction Loss: 0.000060
Epoch [10/10], Reconstruction Loss: 0.000060


In [4]:
print(data.get_samples().shape)
print(data.get_labels().shape)

(3309, 2500)
(3309,)


In [5]:
from sklearn.svm import SVC, LinearSVC

svc = SVC(kernel="rbf", C=100)
svc.fit(data.get_samples()[:3000], data.get_labels()[:3000])
print(svc.score(data.get_samples()[3000:], data.get_labels()[3000:]))

0.6957928802588996


In [6]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate = 0.2)

loops.train(model=model, model_path="autoencoder.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=0, optim="sgd", epochs=10)

loops.test(model_path="autoencoder.pth", test_loader=test_loader)



[INFO] EPOCH: 1/10
Train loss: 0.659312, Train accuracy: 0.6653
[INFO] EPOCH: 2/10
Train loss: 0.640788, Train accuracy: 0.6657
[INFO] EPOCH: 3/10
Train loss: 0.638593, Train accuracy: 0.6657
[INFO] EPOCH: 4/10
Train loss: 0.637796, Train accuracy: 0.6657
[INFO] EPOCH: 5/10
Train loss: 0.638475, Train accuracy: 0.6657
[INFO] EPOCH: 6/10
Train loss: 0.637073, Train accuracy: 0.6657
[INFO] EPOCH: 7/10
Train loss: 0.638133, Train accuracy: 0.6657
[INFO] EPOCH: 8/10
Train loss: 0.637555, Train accuracy: 0.6657
[INFO] EPOCH: 9/10
Train loss: 0.637144, Train accuracy: 0.6657
[INFO] EPOCH: 10/10
Train loss: 0.637568, Train accuracy: 0.6657
[INFO] Testing the model
Test accuracy: 0.6958
